In [1]:
import pandas as pd 
import numpy as np 
from glob import glob
import os
import geopandas as gpd

import csv
import json

from ast import literal_eval

from shapely import wkt

# Set option to display all rows (no truncation)
pd.set_option('display.max_rows', None)

# Set option to display all columns (no truncation)
pd.set_option('display.max_columns', None)

/usr/local/lib/python3.8/dist-packages/geopandas/_compat.py:123: UserWarning: The Shapely GEOS version (3.11.2-CAPI-1.17.2) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(
/tmp/tmp.JwMevPulee/ipykernel_1736555/30998290.py:5: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_p

In [ ]:
# For stats on ncei events, we colocated for each one (see /home/csutter/DRIVE-clean/weather_events/notebooks/stats_events_modelpred.py)
# but curious if they're all the same mapping (which will also help for knowing which regions/cams to run for non-events)

# See if every geometry maps to the same list of cams
- Yes (except BUF / BGM with has overlapping parts of one county, Cauyga, see below)

- This first set of code is using the full events df (not just the events of interest). This was easier just to have one dataset of everything for inspection of location-related columns (checks belwo) and also for connecting to cams on a broader level.
- But for connecting to the events of interest, join by eventid

In [2]:
#### 1 - NCEI data
d_readin = gpd.read_file("/home/csutter/DRIVE-clean/weather_events/data/ncei_events/ncei_ny_events_clean.gpkg")

d_readin.head(4)

# add timedelta duration col back (using duration_sec col)
# note that you have to do this w/ every gdf
d_readin["duration"] = pd.to_timedelta(d_readin["duration_sec"], unit="s")

# may take ~40 seconds

print(len(d_readin))

print(type(d_readin))


7802
<class 'geopandas.geodataframe.GeoDataFrame'>


In [ ]:
d_readin.columns

In [ ]:
print(len(np.unique(d_readin['CZ_NAME'])))
print(len(np.unique(d_readin['WFO'])))
print(len(np.unique(d_readin[['CZ_NAME','WFO']].drop_duplicates())))


In [3]:
### Side quest -- check unique locations | polygons

# print(d_readin.columns)

cols_to_check = ['geometry','CZ_FIPS', 'CZ_NAME', 'WFO','CZ_FIPS_FORMAT', 'ZONE', 'FIPS', 'FIPS_FORMAT']
# Count unique combinations of all columns
comb_ct = len(cols_to_check) 

checks = []
for i in range(0, comb_ct):
    # print("print1: i is")
    # print(i)
    c1 = cols_to_check[i]
    # print("print2")
    # print(c1)
    # print(f"unique {c1}")
    l1 = len(d_readin[[c1]].drop_duplicates())
    # print(l1)
    # print("print3")
    compareto = np.arange(i+1,comb_ct)
    # print("comparing to i's:")
    # print(compareto)
    l2 = []
    for j in compareto:
        # print("print4")
        # print(j)
        c2 = cols_to_check[j]
        # print(c2)
        # print(f"unique COMBINATION {c1} AND {c2}")
        amt2 = len(d_readin[[c1,c2]].drop_duplicates())
        # print(amt2)
        l2.append(amt2)
    checks.append([c1,l1,l2])

print(checks)

# check every pairwise combination - just for reference
# but what we really just care about is that the geometries only map to one unique "qualitative" location, e.g. for polygon A, it always maps to FIP 1, or ZONE XYZ. Which it does, except for the 4th elem (3rd comparison) which is 'WFO'...
# Inspect geometry|WFO...

geoms = d_readin[["geometry"]].drop_duplicates()
geoms_and_wfo = d_readin[["geometry","WFO"]].drop_duplicates()

dups = geoms_and_wfo[geoms_and_wfo.duplicated(subset=['geometry'], keep=False)]

print(dups)

### Asked gemini why the dup WFO:

# Gemini said
# Actually, you’ve stumbled onto a classic NWS "boundary seam." There wasn't necessarily a massive update that moved a county from one to the other, but rather an overlap in responsibility that is baked into the NCEI database for certain Central NY counties.

# The Buffalo (BUF) and Binghamton (BGM) offices share a border that cuts through several "Finger Lakes" zones. Here is why you are seeing both:

# 1. The "Split County" Reality
# In New York, some counties are geographically split between two WFOs for their County Warning Areas (CWA).

# For example, Cayuga County is often the "problem child." The northern half is managed by BUF (Buffalo), while the southern half is managed by BGM (Binghamton).

# However, NCEI often stores the entire county polygon as the geometry. So, if a Winter Storm affects "Cayuga," both BGM and BUF might issue a record for it, and both records will point to the same physical polygon.

###### Conclusion / What to do with it
# since the goal of this code rn is just to match the geometry to cams, don't need to really worry about this. The key thing is that we don't really use WFO as the location identifier to match WFO to Cams - we dont do that. Thus, it shouldn't really be a problem. 


[['geometry', 150, [150, 150, 151, 150, 150, 150, 150]], ['CZ_FIPS', 107, [150, 136, 107, 150, 150, 150]], ['CZ_NAME', 114, [115, 150, 150, 150, 150]], ['WFO', 5, [136, 93, 68, 68]], ['CZ_FIPS_FORMAT', 107, [150, 150, 150]], ['ZONE', 89, [150, 150]], ['FIPS', 63, [63]], ['FIPS_FORMAT', 63, []]]
                                               geometry  WFO
3462  POLYGON ((-76.61650 43.41441, -76.61460 43.386...  BGM
3585  POLYGON ((-76.61650 43.41441, -76.61460 43.386...  BUF


In [4]:
#### 2 - Cam lat and lons (not sure we need, just load data for now)

cams = pd.read_csv("/home/csutter/DRIVE/site_analysis/_reference/511NY_API_GetCameras_response.csv")

cams = cams[((cams["Disabled"]==False)&(cams["Blocked"]==False))]
print(len(cams))

2371


In [5]:

cams_gdf = gpd.GeoDataFrame(
    cams, 
    geometry=gpd.points_from_xy(cams.Longitude, cams.Latitude),
    crs="EPSG:4326" # Start with standard GPS coordinates
)
# print(len(cams_gdf))

cams_gdf = cams_gdf.to_crs(d_readin.crs) # match the CRS to be exactly the system being used in the events dataset
# print(len(cams_gdf))

# Perform the spatial join
events_camloc = gpd.sjoin(
    cams_gdf, 
    d_readin[["EVENT_ID","geometry"]], 
    how="inner",
    predicate="within" # Check if the point is WITHIN the polygon
)

cams_gdf.head(3)

print(len(d_readin[["EVENT_ID","geometry"]]))
print(len(cams))
print(len(events_camloc))

# Splits an event (which was usually one row) into multiple rows, one per each camera, so they larger df size makes sense

# note that doing the join this way (inner) ensures that we're only keeping events and geometries that have cams in them. To get this only for events that we ran (rather than the full d_readin df using right now), see below. Just repeat the steps with joining the cams df only on the events of interest!!

7802
2371
145470


#### Connect to eventsof interest (inner join on EVENT ID) to get just the locations / times we care about

In [6]:

events_ofinterest_paths = ["/home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest/blizzard_allyrs_ceilfloor5min_nobuffer_freq5min.csv",
"/home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest/heavyrain_2425_ceilfloor15min_nobuffer_freq15min.csv",
"/home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest/heavysnow_2025_ceilfloor5min_nobuffer_freq5min.csv",
"/home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest/heavysnow_2425_ceilfloor15min_nobuffer_freq15min.csv",
"/home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest/lakeeffect_2025_ceilfloor15min_nobuffer_freq15min.csv",
"/home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest/lakeeffect_2025_ceilfloor30min_nobuffer_freq30min.csv",
"/home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest/lakeeffect_2425_ceilfloor15min_nobuffer_freq15min.csv",
"/home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest/winterstorm_2425_ceilfloor15min_nobuffer_freq15min.csv",
"/home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest/winterweather_2425_ceilfloor15min_nobuffer_freq15min.csv"]

gdfs_list = []

for ev in events_ofinterest_paths:
    #### 2 - Identified events
    # grab just one file for now, but will need to eventually tie in all of them

    events = pd.read_csv(ev)
    # convert this events df to geopandas df
    # Convert the 'geometry' column from strings to actual Shapely objects
    events['geometry'] = events['geometry'].apply(wkt.loads)
    # Cast as a GeoDataFrame and set the CRS
    # (Use the CRS of your original ncei_gdf, usually "EPSG:4269" for NWS data)
    events_gdf = gpd.GeoDataFrame(events, geometry='geometry', crs="EPSG:4269")
    # Match the points to the NCEI CRS
    # This converts the points' coordinates to fit the storm polygons perfectly
    events_gdf = events_gdf.to_crs(d_readin.crs)
    gdfs_list.append(events_gdf)

eventsofint = pd.concat(gdfs_list,ignore_index=True)
print(type(eventsofint))
print(eventsofint.columns)
print(len(eventsofint))

<class 'geopandas.geodataframe.GeoDataFrame'>
Index(['Unnamed: 0.1', 'Unnamed: 0', 'BEGIN_YEARMONTH', 'BEGIN_DAY',
       'BEGIN_TIME', 'END_YEARMONTH', 'END_DAY', 'END_TIME', 'EPISODE_ID',
       'EVENT_ID', 'STATE', 'STATE_FIPS', 'YEAR', 'MONTH_NAME', 'EVENT_TYPE',
       'CZ_TYPE', 'CZ_FIPS', 'CZ_NAME', 'WFO', 'BEGIN_DATE_TIME',
       'CZ_TIMEZONE', 'END_DATE_TIME', 'INJURIES_DIRECT', 'INJURIES_INDIRECT',
       'DEATHS_DIRECT', 'DEATHS_INDIRECT', 'DAMAGE_PROPERTY', 'DAMAGE_CROPS',
       'SOURCE', 'MAGNITUDE', 'MAGNITUDE_TYPE', 'FLOOD_CAUSE', 'CATEGORY',
       'TOR_F_SCALE', 'TOR_LENGTH', 'TOR_WIDTH', 'TOR_OTHER_WFO',
       'TOR_OTHER_CZ_STATE', 'TOR_OTHER_CZ_FIPS', 'TOR_OTHER_CZ_NAME',
       'BEGIN_RANGE', 'BEGIN_AZIMUTH', 'BEGIN_LOCATION', 'END_RANGE',
       'END_AZIMUTH', 'END_LOCATION', 'BEGIN_LAT', 'BEGIN_LON', 'END_LAT',
       'END_LON', 'EPISODE_NARRATIVE', 'EVENT_NARRATIVE', 'DATA_SOURCE',
       'CZ_FIPS_FORMAT', 'ZONE', 'FIPS', 'FIPS_FORMAT', 'BEGIN_UTC', 'END_UTC',

In [ ]:
events_camloc.columns

In [7]:
# Inner join w the gdf made in the beginning which connects all events to cams

eventsofint_camloc = eventsofint.merge(events_camloc[['Latitude', 'Longitude', 'ID', 'Name','DirectionOfTravel', 'RoadwayName', 'Url', 'VideoUrl', 'Disabled','Blocked','EVENT_ID']], how = "inner", on = "EVENT_ID")

In [ ]:
len(eventsofint_camloc)

In [ ]:
eventsofint_camloc.head(4)

In [ ]:
eventsofint_camloc.columns

In [9]:
# For collecting (polygons) and cams within them, make a df just of the location related cols, including those from the geometry (geom, wfo, etc), as well as those from the cameras (lat, Lon, and ID)

# Use the df from eventsofint_camloc bc these are the locations we will care to compare (like for non-events and buffer events, as in the other sections below)

locs = eventsofint_camloc[['geometry','CZ_FIPS', 'CZ_NAME', 'WFO','CZ_FIPS_FORMAT', 'ZONE', 'FIPS', 'FIPS_FORMAT','Latitude', 'Longitude', 'ID', 'Name','DirectionOfTravel', 'RoadwayName', 'Url', 'VideoUrl', 'Disabled','Blocked']]

print(len(locs))
locsunique = locs.drop_duplicates()

print(len(locsunique))

# Note that this is JUST unique locations - loosely, the structure is geometry | cam-level

locsunique.head(3)
locsunique['ID'] = locsunique['ID'].str.replace('-', '_')



locsunique_bare = locsunique[["geometry","CZ_NAME","WFO", "ID"]] # "Latitude", "Longitude" is already in model pred dfs, dont need it here

locsunique_bare.head(3)


13055
1868


/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


,geometry,CZ_NAME,WFO,ID
0,"MULTIPOLYGON (((-73.42420 40.62371, -73.41857 ...",SOUTHWEST SUFFOLK,OKX,Skyline_1879
1,"MULTIPOLYGON (((-73.42420 40.62371, -73.41857 ...",SOUTHWEST SUFFOLK,OKX,Skyline_1880
2,"MULTIPOLYGON (((-73.42420 40.62371, -73.41857 ...",SOUTHWEST SUFFOLK,OKX,Skyline_1881


## For connecting to non-events (which don't have geometries with them!) 
- But we will eventually want to "connect" event geometries so that we can compare model preds in regions that have events vs when those same regions don't have events
- Uses the df built from events work in cells above


In [ ]:
# Read in non-events model pred data (note, again, there are no "events" df to read in for this! See code /home/csutter/DRIVE-clean/weather_events/notebooks/ncei_dataset_analysis.ipynb for how dates were identified)

In [10]:
inf_sets_ran = ["/home/csutter/DRIVE-clean/operational_runs/set42_nonevents1",
"/home/csutter/DRIVE-clean/operational_runs/set43_nonevents2",
"/home/csutter/DRIVE-clean/operational_runs/set44_nonevents3"]

dirs_w_preds = [f"{i}/data_6_ensembling/*/*/*/*/*" for i in inf_sets_ran]

# print(dirs_w_preds)

pred_datetimes = []
pred_path = []
for dr in dirs_w_preds:
    # print(dr) # just for checking counts per dir
    # dr_files = [] # just for checking counts per dir
    listpredfiles = glob(dr)
    for fl in listpredfiles:
        # print(fl)
        i = fl.rfind("/")
        filedate = fl[i-13:i]
        pred_datetimes.append(filedate)
        # dr_files.append(filedate) # just for checking counts per dir

        ### Must also grab the tracker path which contains the predictions and the image path (should have all the data in it we need)
        pred_path.append(fl)
    # print(len(dr_files)) # just for checking counts per dir


print(len(pred_datetimes))
print(len(pred_path))


print(pred_datetimes[0:4])
print(pred_path[0:4])

605
605
['20221207_0800', '20221207_2200', '20221207_1600', '20221207_1400']
['/home/csutter/DRIVE-clean/operational_runs/set42_nonevents1/data_6_ensembling/2022/12/07/20221207_0800/finalpreds.csv', '/home/csutter/DRIVE-clean/operational_runs/set42_nonevents1/data_6_ensembling/2022/12/07/20221207_2200/finalpreds.csv', '/home/csutter/DRIVE-clean/operational_runs/set42_nonevents1/data_6_ensembling/2022/12/07/20221207_1600/finalpreds.csv', '/home/csutter/DRIVE-clean/operational_runs/set42_nonevents1/data_6_ensembling/2022/12/07/20221207_1400/finalpreds.csv']


In [11]:
### CLEAN UP non-event data first -- make a df and save it out into eventsofinterest

nonweather_date = [x[:8] for x in pred_datetimes]
nonweather_time = [x[9:] for x in pred_datetimes]
nonweather_datetime = pred_datetimes
nonweather_location_str = ["statewide" for x in range(0, len(pred_datetimes))]

print(len(nonweather_date))
print(len(nonweather_time))
print(len(nonweather_datetime))
print(len(nonweather_location_str))

dictfordf = {"nonweather_date":nonweather_date,"nonweather_time":nonweather_time,"nonweather_datetime":nonweather_datetime,"nonweather_location_str":nonweather_location_str }

nonevent_basic_tracker = pd.DataFrame(dictfordf)

nonevent_basic_tracker.head(3)

# Save this out just for reference!!  [ COME BACK TO ]

# Also consider saving out a second varaition for reference -- Consider  joining with the locations df to have this df parsed also by location (kind of like we did in loop code below) [ COME BACK TO ]

605
605
605
605


,nonweather_date,nonweather_time,nonweather_datetime,nonweather_location_str
0,20221207,0800,20221207_0800,statewide
1,20221207,2200,20221207_2200,statewide
2,20221207,1600,20221207_1600,statewide


In [ ]:
# Get stats for nonevents...

# First -- take the model pred df and Connect to locations (wfo, geom, etc.. things in ncei events) that have events associated with them, bc we will ultimately want to compare how locations with events compare to locations with nonevents
# We ran the model preds statewide, but when we merge with locations-with-events, this will shorten the amount of cameras in a model pred file
# Already have the location df with geoms and cam lat/lon
# Connect the model pred df , by site, with the "ID" in that location df

# Second --  for each ""

# Read in all model pred dfs, which have cams in them, and tie in the corresponding region for each cam

# Note the looping order here for nonevents is different than for events, given the different datastructure, have to start with model pred files. And don't need to "find" the right model pred files that match events, bc starting w model pred files.

# Loop 1 - model files
for i in range(0, len(pred_path)):


    d = pd.read_csv(pred_path[i])
    t = pred_datetimes[i]# for tracking results
    # print(len(d))
    # note that im not making the modelpred df into a gdf right now, dont think we need the geom for anything rn at least for stats runs
    d_allinfo = d.merge(locsunique_bare, how = "inner", left_on = "site", right_on = "ID")
    # print(len(d_allinfo))

    # Loop 2 -- by event region, which for us will be CZ_NAME 
    for l in np.unique(d_allinfo["CZ_NAME"]):
        # subset 
        ld1 = d_allinfo[d_allinfo["CZ_NAME"]==l].reset_index()
        # display(ld1.head(2))

        # I think one CZ is split between two WFOs, account for that small detail with one more loop
        for w in np.unique(ld1["WFO"]):

            ld2 = ld1[ld1["CZ_NAME"]==l].reset_index()


            # JUST LOG EVERYTHING NOW THAT IT'S BY REGION / TIME (note: for now, just set episode and event equal to the datetime. Will work out those kinks of how to aggregate them later to make the events/episodes easier to compare to the real events)
            ctsdf = ld2[["select","img_name"]].groupby(["select"]).count().reset_index()

            ctsdf = ctsdf.rename(columns = {"img_name":"count"})

            countdict = ctsdf.set_index("select")["count"].to_dict()

            # display(ctsdf.head(4))
            # print(countdict)

            # other info

            id_event = t
            id_ep = t
            id_type ="nonevent"
            id_loc = l # loop 2
            id_wfo = w # loop 3
            datetime_stamp = t 


            # print(id_event,id_ep,id_type)


            # Your data
            data_to_log = {
                "id_event": id_event, #id_event,
                "id_ep": id_ep,
                "id_type": id_type,
                "id_loc":id_loc,
                "id_wfo":id_wfo,
                "datetime":datetime_stamp,
                "model_counts": countdict # This will be stringified
            }

            csv_path = "/home/csutter/DRIVE-clean/weather_events/models/stats_events_modelpred/stats_nonevents.csv"

            # Pre-process: Convert the nested dict to a JSON string
            data_to_log["model_counts"] = json.dumps(data_to_log["model_counts"])

            # Check if file exists to determine if we need a header
            file_exists = os.path.isfile(csv_path)

            with open(csv_path, mode='a', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=data_to_log.keys())
                
                # Write header only if the file is being created for the first time
                if not file_exists:
                    writer.writeheader()
                    
                writer.writerow(data_to_log)

            # print(f"Logged event {data_to_log['id_event']} to {csv_path}")







In [ ]:
d_allinfo.head(3)

## Connect to buffer preds ran
- Really it's the same code as above for the nonevents, since it was similar in terms of not actually having ncei events as the basis.
- See /home/csutter/DRIVE-clean/weather_events/notebooks/stats_buffer_modelpred.py